Importy potrzebnych modu??w

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.metrics.classification import classification_report, accuracy_score, cohen_kappa_score
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing.label import LabelBinarizer
from tensorflow.python.keras.models import Model
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import concatenate
from tensorflow.keras.models import Sequential
import copy

Odczytanie obrazk?w, zamiana na rozmiar 64x64, zapis do samples i labels

In [ ]:
import cv2
import os
def load_img(indir):
    samples = []
    labels = []
    for class_dir in os.listdir(indir):
        the_class = class_dir
        for file in os.listdir(indir+'/'+class_dir):
            image = cv2.imread("{}/{}/{}".format(indir,class_dir,file))
            image = cv2.resize(image, (64,64))
            samples.append(image)
            labels.append(the_class)
    samples = np.array(samples)
    labels = np.array(labels)
    return samples,labels
samples, labels = load_img('flower_photos')
print('loaded',len(samples),' samples')
print('classes',set(labels))
org_samples = samples
org_labels = labels

Utworzenie modelu CNN

In [ ]:
from tensorflow.keras.layers import Dense, Activation, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
cnnmodel = Sequential()
cnnmodel.add(Conv2D(16, (3, 3), padding="same",input_shape=(64,64,3)))
cnnmodel.add(BatchNormalization())
cnnmodel.add(Activation("relu"))
cnnmodel.add(MaxPooling2D(pool_size=(2, 2)))
cnnmodel.add(Conv2D(32, (3, 3), padding="same"))
cnnmodel.add(Activation("relu"))
cnnmodel.add(MaxPooling2D(pool_size=(2, 2)))
cnnmodel.add(Dropout(0.25))

cnnmodel.add(Flatten())
cnnmodel.add(Dense(512))
cnnmodel.add(Activation("relu"))

cnnmodel.summary()

Zmiana typu labels

In [ ]:
# one-hot labels
from sklearn.preprocessing import LabelBinarizer
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
print("Labels shape",labels.shape)
labels = labels.astype(float)

Przekopiowanie samples do samples2

In [ ]:
samples2 = copy.deepcopy(org_samples)
print('loaded',len(samples2),' samples')

Metody ekstrakcji cech dla samples2

In [ ]:
import mahotas

def fd_hu_moments(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    feature = cv2.HuMoments(cv2.moments(image)).flatten()
    return feature

def fd_haralick(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    haralick = mahotas.features.haralick(gray).mean(axis=0)
    return haralick

def fd_histogram(image, mask=None):
    bins=8
    image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist  = cv2.calcHist([image], [0, 1, 2], None, [bins, bins, bins], [0, 256, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

Stworzenie flat sampli dla modelu Dense

In [ ]:
new_samples2 = []
for i,image in enumerate(samples2):
    fv_hu_moments = fd_hu_moments(image)
    fv_haralick   = fd_haralick(image)
    fv_histogram  = fd_histogram(image)
    if(i%500==0): print(i,'/',len(samples2))
    features = np.hstack([fv_histogram, fv_haralick, fv_hu_moments])
    new_samples2.append(features)
samples2 = np.array(new_samples2)
print(samples2.shape)

Normalizacja do przedzia?u 0 - 1

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
print(np.min(samples2),np.max(samples2))
samples2 = scaler.fit_transform(samples2)
print(np.min(samples2),np.max(samples2))


Utworzenie modelu Dense

In [ ]:
dense_model = Sequential()
dense_model.add(Dense(250, input_dim=532, activation='relu'))
dense_model.add(Dense(250, activation='relu'))

dense_model.summary()
dense_model.compile(loss='categorical_crossentropy', optimizer="adam", metrics=['accuracy'])

Konkatenacja modeli

In [ ]:
combined = concatenate([cnnmodel.output, dense_model.output])
combined = Dense(16, activation="sigmoid")(combined)
combined = Dense(labels.shape[1], activation="sigmoid")(combined)

model = Model(inputs=[cnnmodel.input, dense_model.input], outputs=combined)

print(model.summary())

model.compile(loss='categorical_crossentropy', optimizer="adam",metrics=['accuracy'])

Trenowanie nowo utworzonego modelu

In [ ]:
EPOCHS = 20
BATCH=100
model.fit([samples,samples2], labels, batch_size=BATCH, epochs=EPOCHS)

Osi?gni?te wyniki

In [ ]:
results = model.predict([samples,samples2])
print(confusion_matrix(labels.argmax(axis=1), results.argmax(axis=1)))
print(classification_report(labels.argmax(axis=1), results.argmax(axis=1)))
print("Accuracy: {:.2f}".format(accuracy_score(labels.argmax(axis=1), results.argmax(axis=1))))
print("Cohen's Kappa {:.2f}".format(cohen_kappa_score(labels.argmax(axis=1), results.argmax(axis=1))))